In [1]:
!pip install -q torch torchvision transformers datasets pillow pandas scikit-learn tqdm huggingface_hub matplotlib

In [2]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPImageProcessor, get_cosine_schedule_with_warmup

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

_Config_

In [3]:
DATASET_NAME = "ashraq/fashion-product-images-small"
MODEL_NAME = "openai/clip-vit-base-patch32"
TASKS = [
    "gender",
    "masterCategory",
    "subCategory",
    "articleType",
    "baseColour",
    "season",
    "usage"
]

SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

BATCH_SIZE = 32
EPOCHS = 5

HEAD_LR = 3e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-2

HIDDEN_DIM = 512
DROPOUT = 0.20

UNFREEZE_LAST_N_VISION_LAYERS = 2

USE_CLASS_WEIGHTS = True
USE_AMP = True

MAX_GRAD_NORM = 1.0
EARLY_STOPPING_PATIENCE = 2
NUM_WORKERS = 2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
OUTPUT_DIR = Path("artifacts/models/autocatalogai_clip")
EVAL_DIR = Path("artifacts/evaluation")
PLOT_DIR = Path("artifacts/plots")
PROCESSED_DIR = Path("data/processed")

for directory in [OUTPUT_DIR, EVAL_DIR, PLOT_DIR, PROCESSED_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Device:", DEVICE)
print("Model:", MODEL_NAME)

Device: cuda
Model: openai/clip-vit-base-patch32


In [5]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


set_seed(SEED)

#### Load Full Dataset

In [6]:
raw_dataset = load_dataset(DATASET_NAME, split="train")

print(raw_dataset)
print(raw_dataset.column_names)
print("Total rows:", len(raw_dataset))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/867 [00:00<?, ?B/s]

data/train-00000-of-00002-6cff4c59f91661(…):   0%|          | 0.00/136M [00:00<?, ?B/s]

data/train-00001-of-00002-bb459e5ac5f01e(…):   0%|          | 0.00/135M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/44072 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image'],
    num_rows: 44072
})
['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']
Total rows: 44072


#### Clean Dataset

In [7]:
missing_columns = [task for task in TASKS if task not in raw_dataset.column_names]

if "image" not in raw_dataset.column_names:
    raise ValueError(f"Dataset must contain image column. Found: {raw_dataset.column_names}")

if missing_columns:
    raise ValueError(f"Missing task columns: {missing_columns}")

In [8]:
def is_valid_row(row):
    for task in TASKS:
        value = row.get(task)
        if value is None:
            return False
        if str(value).strip() == "":
            return False
        
    return row.get("image") is not None


clean_dataset = raw_dataset.filter(is_valid_row)
print("Before cleaning:", len(raw_dataset))
print("After cleaning:", len(clean_dataset))

Filter:   0%|          | 0/44072 [00:00<?, ? examples/s]

Before cleaning: 44072
After cleaning: 44072


#### Create Metadata DataFrame

In [9]:
metadata = {}
for_col = []
extra_columns = ["id", "productDisplayName"]

for task in TASKS:
    metadata[task] = [str(value).strip() for value in clean_dataset[task]]


for col in extra_columns:
    if col in clean_dataset.column_names:
        metadata[col] = clean_dataset[col]
        for_col.append(col)

df = pd.DataFrame(metadata)
df["dataset_idx"] = np.arange(len(clean_dataset))

df.head()

,gender,masterCategory,subCategory,articleType,baseColour,season,usage,id,productDisplayName,dataset_idx
0,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,Casual,15970,Turtle Check Men Navy Blue Shirt,0
1,Men,Apparel,Bottomwear,Jeans,Blue,Summer,Casual,39386,Peter England Men Party Blue Jeans,1
2,Women,Accessories,Watches,Watches,Silver,Winter,Casual,59263,Titan Women Silver Watch,2
3,Men,Apparel,Bottomwear,Track Pants,Black,Fall,Casual,21379,Manchester United Men Solid Black Track Pants,3
4,Men,Apparel,Topwear,Tshirts,Grey,Summer,Casual,53759,Puma Men Grey T-shirt,4


#### Label Distribution

In [10]:
label_distribution = {}

for task in TASKS:
    counts = df[task].value_counts().to_dict()
    label_distribution[task] = counts
    
with open(EVAL_DIR / "label_distribution.json", "w", encoding="utf-8") as f:
    json.dump(label_distribution, f, indent=2, ensure_ascii=False)

In [11]:
summary = {
    "dataset_name": DATASET_NAME,
    "total_clean_samples": len(df),
    "tasks": TASKS,
    "num_classes": {
        task: int(df[task].nunique())
        for task in TASKS
    }
}

with open(EVAL_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

summary

{'dataset_name': 'ashraq/fashion-product-images-small',
 'total_clean_samples': 44072,
 'tasks': ['gender',
  'masterCategory',
  'subCategory',
  'articleType',
  'baseColour',
  'season',
  'usage'],
 'num_classes': {'gender': 5,
  'masterCategory': 7,
  'subCategory': 45,
  'articleType': 141,
  'baseColour': 46,
  'season': 4,
  'usage': 8}}

#### Train / Validation / Test Split

In [12]:
def make_safe_stratify_labels(series):
    counts = series.value_counts()
    return series.apply(lambda x: x if counts[x] >= 2 else "__rare__")

stratify_labels = make_safe_stratify_labels(df["articleType"])
all_indices = df.index.to_numpy()

In [13]:
train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=SEED,
    stratify=stratify_labels
)

temp_df = df.loc[temp_idx].copy()
temp_stratify_labels = make_safe_stratify_labels(temp_df["articleType"])

In [14]:
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_stratify_labels
)

train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()
test_df = df.loc[test_idx].copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "test"

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

In [15]:
print("Train:", len(train_df), round(len(train_df) / len(df), 3))
print("Validation:", len(val_df), round(len(val_df) / len(df), 3))
print("Test:", len(test_df), round(len(test_df) / len(df), 3))

Train: 30850 0.7
Validation: 6611 0.15
Test: 6611 0.15


#### Build HF Split Datasets

In [16]:
train_hf_dataset = clean_dataset.select(train_df["dataset_idx"].tolist())
val_hf_dataset = clean_dataset.select(val_df["dataset_idx"].tolist())
test_hf_dataset = clean_dataset.select(test_df["dataset_idx"].tolist())

len(train_hf_dataset), len(val_hf_dataset), len(test_hf_dataset)

(30850, 6611, 6611)